In [1]:
import os
import math
import json
import numpy as np
import pandas as pd
from PIL import Image
import astropy.io.fits as fits
import matplotlib.cm as cm
import warnings
import base64
from io import BytesIO

warnings.filterwarnings('ignore', category=RuntimeWarning)

# ==========================================
# SETTINGS
# ==========================================
DF_PATH = '../MANGA/hydra-hc8-manga/hydra_tables/hydra1.01.csv'
IMG_DIR = '../MANGA/png'

# Alterado para salvar tudo diretamente na pasta 'MaNGA' para o GitHub Pages
OUTPUT_DIR = 'MaNGA'
THUMBS_DIR = os.path.join(OUTPUT_DIR, 'thumbs')

COL_ID   = "MANGA_ID"
COL_X    = "PC1"
COL_Y    = "PC2"

COLOR_OPTIONS = ["Ttype", "A3", "C1", "sigma_psi", "Ci", "Cm", "Co", "Cd", "Ck", "H"]
IMAGE_SUFFIXES = ['.png', '_r.fits', '_g.fits', '.fits', '.fit']

PAGE_TITLE = "Galaxy map in the PC1-PC2 plane"
SAMPLE_NAME = "MaNGA sample"
MOSAIC_FILENAME = "galaxy_mosaic.png"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(THUMBS_DIR, exist_ok=True)

print("📂 Lendo tabela de dados...")
df = pd.read_csv(DF_PATH)
N = len(df)
print(f"-> Total de galáxias na tabela: {N}")

# ==========================================
# MOSAIC GRID CALCULATION
# ==========================================
print("🗺️ Calculando o grid do mosaico...")
x_min, x_max = df[COL_X].min(), df[COL_X].max()
y_min, y_max = df[COL_Y].min(), df[COL_Y].max()
x_range = x_max - x_min + 1e-9
y_range = y_max - y_min + 1e-9
aspect = x_range / y_range

folga = 2.0
usable_GRID_Y = int(math.ceil(math.sqrt((N * folga) / aspect)))
usable_GRID_X = int(math.ceil(usable_GRID_Y * aspect))

margem_x = int(usable_GRID_X * 0.25)
margem_y = int(usable_GRID_Y * 0.25)

GRID_X = usable_GRID_X + (2 * margem_x)
GRID_Y = usable_GRID_Y + (2 * margem_y)
THUMB_SIZE = 50

df_plot = df.copy()
df_plot['dist_center'] = (df_plot[COL_X] - df_plot[COL_X].mean())**2 + (df_plot[COL_Y] - df_plot[COL_Y].mean())**2
df_plot = df_plot.sort_values('dist_center')

max_radius = max(GRID_X, GRID_Y)
search_limit = min(max_radius, 60)
spiral_offsets = [(dx, dy, dx**2 + dy**2) for dx in range(-search_limit, search_limit + 1) for dy in range(-search_limit, search_limit + 1)]
spiral_offsets.sort(key=lambda x: x[2])
spiral_offsets = [(dx, dy) for dx, dy, dist in spiral_offsets]

grid_mask = np.zeros((GRID_Y, GRID_X), dtype=bool)
df_plot['grid_i'] = -1
df_plot['grid_j'] = -1

for idx, row in df_plot.iterrows():
    ideal_i = int(round(((row[COL_X] - x_min) / x_range) * (usable_GRID_X - 1))) + margem_x
    ideal_j = int(round(((row[COL_Y] - y_min) / y_range) * (usable_GRID_Y - 1))) + margem_y

    for dx, dy in spiral_offsets:
        ni, nj = ideal_i + dx, ideal_j + dy
        if 0 <= ni < GRID_X and 0 <= nj < GRID_Y:
            if not grid_mask[nj, ni]:
                grid_mask[nj, ni] = True
                df_plot.at[idx, 'grid_i'] = ni
                df_plot.at[idx, 'grid_j'] = nj
                break

df_plot = df_plot[df_plot['grid_i'] != -1]
print(f"-> Grid posicionado com sucesso! {len(df_plot)} galáxias mapeadas.")

# ==========================================
# IMAGE PROCESSING
# ==========================================
print("🖼️ Processando imagens e gerando miniaturas em 'MaNGA/thumbs/'...")
def process_and_save_image(obj_id, target_size):
    for suffix in IMAGE_SUFFIXES:
        path = os.path.join(IMG_DIR, f"{obj_id}{suffix}")
        if not os.path.exists(path): continue
        try:
            img = None
            if suffix.lower().endswith('.png'):
                img = Image.open(path).convert("RGBA")
            elif suffix.lower().endswith(('.fits', '.fit')):
                with fits.open(path) as hdul:
                    for hdu in hdul:
                        if hdu.data is not None and len(hdu.data.shape) == 2:
                            data = hdu.data
                            vmin, vmax = np.nanpercentile(data, [15, 99.5])
                            norm_data = np.clip((data - vmin) / (vmax - vmin + 1e-8), 0, 1)
                            img_array = cm.gray(np.arcsinh(10.0 * norm_data) / np.arcsinh(10.0), bytes=True)
                            img = Image.fromarray(img_array).convert("RGBA")
                            break
            if img is not None:
                img_resized = img.resize((target_size, target_size), Image.Resampling.LANCZOS)
                out_path = os.path.join(THUMBS_DIR, f"{obj_id}.png")
                img_resized.save(out_path, format="PNG")
                return img_resized
        except Exception:
            pass
    return None

canvas = np.zeros((GRID_Y * THUMB_SIZE, GRID_X * THUMB_SIZE, 4), dtype=np.uint8)
galaxy_records = {}

processed_count = 0
total_to_process = len(df_plot)

for idx, row in df_plot.iterrows():
    i, j = int(row['grid_i']), int(row['grid_j'])
    obj_id = str(row[COL_ID])

    img_pil = process_and_save_image(obj_id, target_size=120)

    meta = {
        "id": obj_id,
        "x": float(row[COL_X]),
        "y": float(row[COL_Y]),
        # Caminho relativo correto para o JSON carregar de dentro da pasta MaNGA
        "img": f"thumbs/{obj_id}.png"
    }
    for col in COLOR_OPTIONS:
        val = row[col] if col in row and not pd.isna(row[col]) else "N/A"
        meta[col] = float(val) if isinstance(val, (int, float, np.number)) else str(val)

    galaxy_records[f"{i},{j}"] = meta

    if img_pil is not None:
        processed_count += 1
        y_start = (GRID_Y - 1 - j) * THUMB_SIZE
        y_end = y_start + THUMB_SIZE
        x_start = i * THUMB_SIZE
        x_end = x_start + THUMB_SIZE
        img_np = np.array(img_pil.resize((THUMB_SIZE, THUMB_SIZE), Image.Resampling.LANCZOS))
        if img_np.shape[2] == 4:
            alpha = img_np[:, :, 3] / 255.0
            for c in range(3):
                canvas[y_start:y_end, x_start:x_end, c] = (1 - alpha) * canvas[y_start:y_end, x_start:x_end, c] + alpha * img_np[:, :, c]
            canvas[y_start:y_end, x_start:x_end, 3] = np.maximum(canvas[y_start:y_end, x_start:x_end, 3], img_np[:, :, 3])
        else:
            canvas[y_start:y_end, x_start:x_end, :3] = img_np[:, :, :3]
            canvas[y_start:y_end, x_start:x_end, 3] = 255

    if processed_count % 50 == 0:
        print(f"-> Processadas {processed_count}/{total_to_process} imagens...")

print("📦 Compactando mosaico principal...")
output_path = os.path.join(OUTPUT_DIR, MOSAIC_FILENAME)
img_canvas_pil = Image.fromarray(canvas)
img_canvas_pil.save(output_path, format="PNG")
print(f"✅ Map saved to PNG: {output_path}")

# Salvar também em PDF na pasta MaNGA
pdf_path = os.path.join(OUTPUT_DIR, MOSAIC_FILENAME.replace(".png", "") + ".pdf")
background = Image.new("RGB", img_canvas_pil.size, (255, 255, 255))
background.paste(img_canvas_pil, mask=img_canvas_pil.split()[3])
background.save(pdf_path, format="PDF")
print(f"✅ Map saved to PDF: {pdf_path}")

buffered = BytesIO()
img_canvas_pil.save(buffered, format="PNG")
mosaic_base64 = f"data:image/png;base64,{base64.b64encode(buffered.getvalue()).decode()}"

output_data = {
    "title": PAGE_TITLE,
    "sample": SAMPLE_NAME,
    "n_galaxies": int(len(df_plot)),
    "col_x": COL_X,
    "col_y": COL_Y,
    "properties": COLOR_OPTIONS,
    "grid_x": GRID_X,
    "grid_y": GRID_Y,
    "mosaic_base64": mosaic_base64,
    "xs": (df_plot['grid_i'] + 0.5).tolist(),
    "ys": (df_plot['grid_j'] + 0.5).tolist(),
    "ids": df_plot[COL_ID].astype(str).tolist(),
    "records": galaxy_records
}

print("💾 Salvando o arquivo 'galeria_data.json'...")
with open(os.path.join(OUTPUT_DIR, "galeria_data.json"), "w", encoding="utf-8") as f:
    json.dump(output_data, f)

# ==========================================
# index.html (Geração do Front-end)
# ==========================================
html_content = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Galaxy map</title>
<script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
<style>
  :root {
    --bg: #ffffff;
    --panel: #f8f9fa;
    --border: #dee2e6;
    --text: #212529;
    --muted: #6c757d;
    --accent: #1f77b4;
    --sel: #d62728;
    --nb: #1f77b4;
  }
  * { box-sizing: border-box; }
  html, body { height: 100%; margin: 0; }
  body {
    background: var(--bg); color: var(--text);
    font-family: -apple-system, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
    font-size: 14px; display: flex; flex-direction: column;
  }
  header {
    padding: 10px 18px; border-bottom: 1px solid var(--border);
    display: flex; align-items: baseline; gap: 14px; flex-wrap: wrap;
  }
  header h1 { font-size: 17px; font-weight: 600; margin: 0; }
  header .sub { color: var(--muted); font-size: 13px; }
  main { flex: 1; display: flex; min-height: 0; }
  #plot-area { flex: 1; min-width: 0; background: #000; }
  aside {
    width: 360px; flex: none; background: var(--panel);
    border-left: 1px solid var(--border); padding: 16px; overflow-y: auto;
  }
  aside h2 {
    font-size: 12px; font-weight: 600; text-transform: uppercase;
    letter-spacing: .05em; color: var(--muted); margin: 0 0 8px;
  }
  .section + .section { margin-top: 20px; }
  #empty { color: var(--muted); line-height: 1.5; }
  .sel-head { display: flex; gap: 12px; align-items: center; margin-bottom: 10px; }
  .sel-head img {
    width: 96px; height: 96px; object-fit: cover; background: #000;
    border: 1px solid var(--border); border-radius: 4px;
  }
  .sel-head .name { font-size: 16px; font-weight: 600; }
  .sel-head .pos { color: var(--muted); font-size: 12.5px; margin-top: 2px; }
  table { width: 100%; border-collapse: collapse; background: #fff; border: 1px solid var(--border); }
  td { padding: 4px 8px; border-bottom: 1px solid var(--border); }
  tr:last-child td { border-bottom: none; }
  td:first-child { color: var(--muted); width: 40%; }
  td:last-child { text-align: right; font-variant-numeric: tabular-nums; }
  .grid { display: grid; grid-template-columns: repeat(5, 1fr); gap: 4px; }
  .grid img {
    width: 100%; aspect-ratio: 1; object-fit: cover; display: block; background: #000;
    border: 1px solid var(--border); border-radius: 3px; cursor: pointer;
  }
  .grid img:hover { border-color: var(--accent); }
  .note { color: var(--muted); font-size: 12px; line-height: 1.45; margin-top: 18px; }
  @media (max-width: 900px) {
    main { flex-direction: column; }
    #plot-area { flex: none; height: 60vh; }
    aside { width: 100%; border-left: none; border-top: 1px solid var(--border); }
  }
</style>
</head>
<body>
<header>
  <h1 id="title">Galaxy map</h1>
  <span class="sub" id="subtitle"></span>
</header>
<main>
  <div id="plot-area"></div>
  <aside>
    <div class="section">
      <h2>Selected galaxy</h2>
      <div id="info"><div id="empty">Click a galaxy in the map to see its measured properties.</div></div>
    </div>
    <div class="section" id="nb-section" style="display:none">
      <h2>25 nearest galaxies in PC1-PC2</h2>
      <div class="grid" id="neighbors"></div>
    </div>
    <p class="note">
      Each galaxy is placed on a grid at its PC1-PC2 position. Cells are moved
      slightly to avoid overlap, so positions are approximate. Use the scroll
      wheel to zoom and drag to pan.
    </p>
  </aside>
</main>

<script>
const LABELS = {
  T: 'T', A3: 'A<sub>3</sub>', C1: 'C<sub>1</sub>', sigma_psi: '&sigma;<sub>&psi;</sub>',
  Ci: 'C<sub>i</sub>', Cm: 'C<sub>m</sub>', Co: 'C<sub>o</sub>',
  Cd: 'C<sub>d</sub>', Ck: 'C<sub>k</sub>', H: 'H'
};
const fmt = v => (typeof v === 'number') ? v.toFixed(3) : v;

fetch('galeria_data.json').then(r => r.json()).then(data => {
  const rec = data.records;
  const keys = Object.keys(rec);

  document.getElementById('title').textContent = data.title;
  document.title = data.title;
  document.getElementById('subtitle').textContent =
    data.sample + ', ' + data.n_galaxies + ' galaxies. Click a galaxy to select it.';

  const trace = {
    x: data.xs, y: data.ys, text: data.ids, mode: 'markers',
    marker: { size: 14, opacity: 0 },
    hovertemplate: '%{text}<extra></extra>'
  };

  const layout = {
    margin: { l: 0, r: 0, t: 0, b: 0 },
    plot_bgcolor: '#000', paper_bgcolor: '#000',
    xaxis: { showgrid: false, zeroline: false, visible: false, range: [0, data.grid_x] },
    yaxis: { showgrid: false, zeroline: false, visible: false, range: [0, data.grid_y],
             scaleanchor: 'x', scaleratio: 1 },
    images: [{
      source: data.mosaic_base64, xref: 'x', yref: 'y',
      x: 0, y: data.grid_y, sizex: data.grid_x, sizey: data.grid_y,
      sizing: 'stretch', layer: 'below'
    }],
    annotations: [
      { xref: 'paper', yref: 'paper', x: 0.01, y: 0.01, xanchor: 'left', yanchor: 'bottom',
        text: data.col_x + ' \u2192', showarrow: false, font: { color: '#fff', size: 13 } },
      { xref: 'paper', yref: 'paper', x: 0.01, y: 0.05, xanchor: 'left', yanchor: 'bottom',
        text: data.col_y + ' \u2191', showarrow: false, font: { color: '#fff', size: 13 } }
    ],
    shapes: [],
    hoverlabel: { bgcolor: '#fff', font: { color: '#000' } }
  };

  const config = {
    responsive: true, displaylogo: false, scrollZoom: true,
    modeBarButtonsToRemove: ['select2d', 'lasso2d', 'autoScale2d'],
    toImageButtonOptions: { format: 'png', filename: 'galaxy_map', scale: 3 }
  };

  const plot = document.getElementById('plot-area');
  Plotly.newPlot(plot, [trace], layout, config);

  const box = (key, color, width) => {
    const [i, j] = key.split(',').map(Number);
    return { type: 'rect', xref: 'x', yref: 'y', x0: i, x1: i + 1, y0: j, y1: j + 1,
             line: { color: color, width: width }, fillcolor: 'rgba(0,0,0,0)' };
  };

  function select(key) {
    const g = rec[key];
    if (!g) return;

    let rows = '';
    data.properties.forEach(p => {
      if (p in g) rows += '<tr><td>' + (LABELS[p] || p) + '</td><td>' + fmt(g[p]) + '</td></tr>';
    });
    document.getElementById('info').innerHTML =
      '<div class="sel-head"><img src="' + g.img + '" alt="' + g.id + '">' +
      '<div><div class="name">' + g.id + '</div>' +
      '<div class="pos">' + data.col_x + ' = ' + g.x.toFixed(2) + '<br>' +
      data.col_y + ' = ' + g.y.toFixed(2) + '</div></div></div>' +
      '<table>' + rows + '</table>';

    const d = keys.map(k => ({ k: k,
      d: (rec[k].x - g.x) ** 2 + (rec[k].y - g.y) ** 2 }));
    d.sort((a, b) => a.d - b.d);
    const near = d.filter(o => o.k !== key).slice(0, 25);

    document.getElementById('neighbors').innerHTML = near.map(o =>
      '<img src="' + rec[o.k].img + '" title="' + rec[o.k].id + '" data-key="' + o.k + '">'
    ).join('');
    document.getElementById('nb-section').style.display = 'block';

    Plotly.relayout(plot, { shapes:
      near.map(o => box(o.k, '#4da3ff', 1)).concat([box(key, '#d62728', 3)]) });
  }

  plot.on('plotly_click', ev => {
    const pt = ev.points[0];
    select(Math.floor(pt.x) + ',' + Math.floor(pt.y));
  });
  document.getElementById('neighbors').addEventListener('click', e => {
    if (e.target.dataset.key) select(e.target.dataset.key);
  });
});
</script>
</body>
</html>
"""

print("💾 Salvando o arquivo 'index.html' na pasta MaNGA...")
with open(os.path.join(OUTPUT_DIR, "index.html"), "w", encoding="utf-8") as f:
    f.write(html_content)

print("\n==========================================")
print("🎉 TUDO PRONTO COM SUCESSO!")
print("==========================================")
print("-> Pasta 'MaNGA/' criada/atualizada com sucesso.")
print("-> index.html e galeria_data.json salvos dentro de 'MaNGA/'.")
print("-> galaxy_mosaic.png e galaxy_mosaic.pdf salvos dentro de 'MaNGA/'.")
print("-> Pasta 'MaNGA/thumbs/' preenchida com as miniaturas.")

📂 Lendo tabela de dados...
-> Total de galáxias na tabela: 5914
🗺️ Calculando o grid do mosaico...
-> Grid posicionado com sucesso! 5914 galáxias mapeadas.
🖼️ Processando imagens e gerando miniaturas em 'MaNGA/thumbs/'...
-> Processadas 50/5914 imagens...
-> Processadas 100/5914 imagens...
-> Processadas 150/5914 imagens...
-> Processadas 200/5914 imagens...
-> Processadas 250/5914 imagens...
-> Processadas 300/5914 imagens...
-> Processadas 350/5914 imagens...
-> Processadas 400/5914 imagens...
-> Processadas 450/5914 imagens...
-> Processadas 500/5914 imagens...
-> Processadas 550/5914 imagens...
-> Processadas 600/5914 imagens...
-> Processadas 650/5914 imagens...
-> Processadas 700/5914 imagens...
-> Processadas 750/5914 imagens...
-> Processadas 800/5914 imagens...
-> Processadas 850/5914 imagens...
-> Processadas 900/5914 imagens...
-> Processadas 950/5914 imagens...
-> Processadas 1000/5914 imagens...
-> Processadas 1050/5914 imagens...
-> Processadas 1100/5914 imagens...
-> Pro

In [34]:
# --- WORKAROUND PARA O ERRO DO MATPLOTLIB ---
import matplotlib
from matplotlib import RcParams
if not hasattr(RcParams, '_get'):
    RcParams._get = RcParams.get
# ---------------------------------------------

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
from matplotlib.patches import ConnectionPatch
from PIL import Image

# ==========================================
# CONFIGURAÇÕES (MaNGA ou EFIGI)
# ==========================================
BASE_DIR = 'MaNGA'
JSON_PATH = os.path.join(BASE_DIR, 'galeria_data.json')
MOSAIC_PATH = os.path.join(BASE_DIR, 'galaxy_mosaic.png')
OUTPUT_PDF = os.path.join(BASE_DIR, 'mapa_visual_extremos_pub.pdf')

print("📂 Carregando dados...")
mosaic_img = Image.open(MOSAIC_PATH).convert('RGBA')
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

GRID_X = data['grid_x']
GRID_Y = data['grid_y']
THUMB_SIZE = mosaic_img.width / GRID_X

galaxies = []
for key, val in data['records'].items():
    i, j = map(int, key.split(','))
    val['i'] = i
    val['j'] = j
    val['x_px'] = i * THUMB_SIZE
    val['y_px'] = (GRID_Y - 1 - j) * THUMB_SIZE
    galaxies.append(val)

print("🔍 Selecionando as top 10 galáxias extremas para cada canto...")
tl_gals = sorted(galaxies, key=lambda g: g['x'] - g['y'])[:10]
tr_gals = sorted(galaxies, key=lambda g: -(g['x'] + g['y']))[:10]
bl_gals = sorted(galaxies, key=lambda g: g['x'] + g['y'])[:10]
br_gals = sorted(galaxies, key=lambda g: -(g['x'] - g['y']))[:10]

print("🎨 Gerando figura de publicação com nomes das galáxias nas miniaturas...")
fig, ax = plt.subplots(figsize=(32, 26), facecolor='white')
ax.set_facecolor('white')
ax.imshow(mosaic_img)
ax.axis('off')

img_w, img_h = mosaic_img.size
margin_x = img_w * 0.15
margin_y = img_h * 0.16
ax.set_xlim(-margin_x, img_w + margin_x)
ax.set_ylim(img_h + margin_y, -margin_y)

# Cores exclusivas por canto
color_tl = '#1f77b4'  # Azul
color_tr = '#d62828'  # Vermelho
color_bl = '#2ca02c'  # Verde
color_br = '#9467bd'  # Roxo
line_alpha = 0.45

def draw_group_bounding_box(gals, color):
    xs = [g['x_px'] for g in gals]
    ys = [g['y_px'] for g in gals]
    min_x, max_x = min(xs), max(xs) + THUMB_SIZE
    min_y, max_y = min(ys), max(ys) + THUMB_SIZE
    
    rect = patches.Rectangle(
        (min_x, min_y), max_x - min_x, max_y - min_y,
        linewidth=2.5, edgecolor=color, facecolor='none', linestyle='--', zorder=4
    )
    ax.add_patch(rect)
    
    for g in gals:
        r_inner = patches.Rectangle(
            (g['x_px'], g['y_px']), THUMB_SIZE, THUMB_SIZE,
            linewidth=2.2, edgecolor=color, facecolor='none', zorder=5
        )
        ax.add_patch(r_inner)

draw_group_bounding_box(tl_gals, color_tl)
draw_group_bounding_box(tr_gals, color_tr)
draw_group_bounding_box(bl_gals, color_bl)
draw_group_bounding_box(br_gals, color_br)

# ----------------------------------------------------
# PAINEL TETRIS DE 10 GALÁXIAS (Com nomes impressos nas imagens)
# ----------------------------------------------------
def add_inset_tetris_10(gals, position, title, align='left', is_bottom=False, border_color='#333333'):
    ax_in = ax.inset_axes(position)
    
    canvas = Image.new('RGBA', (2400, 2400), color=(255, 255, 255, 0))
    
    # Vamos armazenar pares (imagem PIL, ID da galáxia)
    items_to_draw = []
    for g in gals:
        img_path = os.path.join(BASE_DIR, g['img']) if 'img' in g else os.path.join(BASE_DIR, f"thumbs/{g['id']}.png")
        if os.path.exists(img_path):
            img_pil = Image.open(img_path).convert('RGBA').resize((600, 600), Image.Resampling.LANCZOS)
        else:
            img_pil = Image.new('RGBA', (600, 600), color=(0, 0, 0, 255))
        
        galaxy_id = str(g.get('id', g.get('name', 'galaxy'))).strip()
        items_to_draw.append((img_pil, galaxy_id))
    
    if not is_bottom and align == 'left':
        coords = [
            (0, 0), (600, 0), (1200, 0), (1800, 0),     # Base superior (4)
            (0, 600), (600, 600), (1200, 600),          # Meio (3)
            (0, 1200), (600, 1200),                     # Meio superior (2)
            (0, 1800)                                   # Ponta inferior (1)
        ]
    elif not is_bottom and align == 'right':
        coords = [
            (0, 0), (600, 0), (1200, 0), (1800, 0),     # Base superior (4)
            (600, 600), (1200, 600), (1800, 600),       # Meio (3)
            (1200, 1200), (1800, 1200),                 # Meio superior (2)
            (1800, 1800)                                # Ponta inferior (1)
        ]
    elif is_bottom and align == 'left':
        coords = [
            (0, 0),                                     # Ponta superior (1)
            (0, 600), (600, 600),                       # Meio superior (2)
            (0, 1200), (600, 1200), (1200, 1200),       # Meio (3)
            (0, 1800), (600, 1800), (1200, 1800), (1800, 1800) # Base inferior (4)
        ]
    else:
        coords = [
            (1800, 0),                                  # Ponta superior (1)
            (1200, 600), (1800, 600),                   # Meio superior (2)
            (600, 1200), (1200, 1200), (1800, 1200),    # Meio (3)
            (0, 1800), (600, 1800), (1200, 1800), (1800, 1800) # Base inferior (4)
        ]
        
    for (img_pil, gid), (x, y) in zip(items_to_draw, coords):
        canvas.paste(img_pil, (x, y))
        
    ax_in.imshow(np.array(canvas))
    ax_in.axis('off')
    
    # Adicionar nome de cada galáxia em cima da miniatura e desenhar bordas
    for idx, (x, y) in enumerate(coords):
        rect_x = x / 2400.0
        rect_y = 1.0 - (y + 600) / 2400.0
        rect = patches.Rectangle((rect_x, rect_y), 600/2400.0, 600/2400.0, transform=ax_in.transAxes, 
                                 linewidth=2.0, edgecolor=border_color, facecolor='none')
        ax_in.add_patch(rect)
        
        # Texto com ID/nome da galáxia dentro de cada célula do inset
        gid = items_to_draw[idx][1]
        text_x_pos = (x + 30) / 2400.0
        text_y_pos = 1.0 - (y + 50) / 2400.0
        
        stroke = [path_effects.withStroke(linewidth=2.5, foreground='black')]
        ax_in.text(text_x_pos, text_y_pos, gid, transform=ax_in.transAxes, color='white',
                   fontsize=20, fontweight='bold', va='top', ha='left', path_effects=stroke)

    # Textos mais próximos dos blocos
    y_text = -0.03 if is_bottom else 1.03
    va = 'top' if is_bottom else 'bottom'
    x_text = 0.0 if align == 'left' else 1.0
    
    ax_in.text(x_text, y_text, title,
               transform=ax_in.transAxes, color='black',
               fontsize=44, fontweight='bold',
               ha=align, va=va)
               
    return ax_in, coords

# Insets com tamanhos maiores para acomodar as miniaturas ampliadas
width, height = 0.42, 0.42 
ax_tl, coords_tl = add_inset_tetris_10(tl_gals, [0, 0.56, width, height], "Low PC1, High PC2", align='left', is_bottom=False, border_color=color_tl)
ax_tr, coords_tr = add_inset_tetris_10(tr_gals, [0.60, 0.56, width, height], "High PC1, High PC2", align='right', is_bottom=False, border_color=color_tr)
ax_bl, coords_bl = add_inset_tetris_10(bl_gals, [0, 0.01, width, height], "Low PC1, Low PC2", align='left', is_bottom=True, border_color=color_bl)
ax_br, coords_br = add_inset_tetris_10(br_gals, [0.60, 0.01, width, height], "High PC1, Low PC2", align='right', is_bottom=True, border_color=color_br)

# ----------------------------------------------------
# CONEXÕES DIRETAS
# ----------------------------------------------------
def connect_tetris_10_thumbnails(ax_in, gals, coords, is_top_panel, is_left_panel, line_color):
    rad_val = 0.15 if is_left_panel else -0.15
    if is_top_panel: rad_val *= -1
    
    for idx, (g, (cx, cy)) in enumerate(zip(gals, coords)):
        x_fraction = (cx + 300) / 2400.0
        y_fraction = 1.0 - (cy + 300) / 2400.0
        
        target_x = g['x_px'] + THUMB_SIZE / 2
        target_y = g['y_px'] + THUMB_SIZE / 2
        
        con = ConnectionPatch(
            xyA=(x_fraction, y_fraction), coordsA="axes fraction",
            xyB=(target_x, target_y), coordsB="data",
            axesA=ax_in, axesB=ax,
            color=line_color, linewidth=1.4, 
            connectionstyle=f"arc3,rad={rad_val}",
            alpha=line_alpha
        )
        fig.add_artist(con)

connect_tetris_10_thumbnails(ax_tl, tl_gals, coords_tl, is_top_panel=True, is_left_panel=True, line_color=color_tl)
connect_tetris_10_thumbnails(ax_tr, tr_gals, coords_tr, is_top_panel=True, is_left_panel=False, line_color=color_tr)
connect_tetris_10_thumbnails(ax_bl, bl_gals, coords_bl, is_top_panel=False, is_left_panel=True, line_color=color_bl)
connect_tetris_10_thumbnails(ax_br, br_gals, coords_br, is_top_panel=False, is_left_panel=False, line_color=color_br)

# ----------------------------------------------------
# EIXOS GIGANTES DE PONTA A PONTA
# ----------------------------------------------------
ax.annotate('', xy=(img_w, img_h * 1.04), xytext=(0, img_h * 1.04),
            arrowprops=dict(arrowstyle="<->", color="black", lw=3.5))
ax.text(img_w * 0.5, img_h * 1.08, 'PC1', color='black', ha='center', va='bottom', fontsize=48, fontweight='bold')

ax.annotate('', xy=(img_w, -0.03 * img_h), xytext=(0, -0.03 * img_h),
            arrowprops=dict(arrowstyle="<->", color="black", lw=3.5))

ax.annotate('', xy=(-0.04 * img_w, img_h), xytext=(-0.04 * img_w, 0),
            arrowprops=dict(arrowstyle="<->", color="black", lw=3.5))
ax.text(-0.08 * img_w, img_h * 0.5, 'PC2', color='black', ha='center', va='center', fontsize=48, fontweight='bold', rotation=90)

ax.annotate('', xy=(img_w * 1.02, img_h), xytext=(img_w * 1.02, 0),
            arrowprops=dict(arrowstyle="<->", color="black", lw=3.5))

# ==========================================
# SALVAR PDF FINAL
# ==========================================
print(f"💾 Salvando o PDF final em {OUTPUT_PDF}...")
plt.savefig(OUTPUT_PDF, facecolor='white', bbox_inches='tight', dpi=300)
plt.close()

print("✅ Concluído! Nomes das galáxias adicionados em cada miniatura com sucesso.")

📂 Carregando dados...
🔍 Selecionando as top 10 galáxias extremas para cada canto...
🎨 Gerando figura de publicação com nomes das galáxias nas miniaturas...
💾 Salvando o PDF final em MaNGA/mapa_visual_extremos_pub.pdf...
✅ Concluído! Nomes das galáxias adicionados em cada miniatura com sucesso.
